## Week 11 Practice

### Problem 1: Implementing an objective function in NumPy and PyTorch
Implement two Python functions that implements the mathematical functions $f(x) = 3x^\intercal x - x_1 - 4$. One function should work for $x$ a `numpy.ndarray` and the other should work for $x$ a `torch.FloatTensor`.

In [23]:
import torch
import numpy as np


def f_numpy(x):
    # f(x) = 3 x^T x - x_1 - 4
    return float(3 * np.dot(x, x) - x[0] - 4)

def f_torch(x):
    # f(x) = 3 x^T x - x_1 - 4
    # .item() converts a one-element tensor to a standard Python number
    return (3 * torch.dot(x, x) - x[0] - 4).item()


In [24]:
assert abs(f_numpy(np.zeros(4)) + 4) < 1e-6
assert abs(f_numpy(np.ones(2)) - 1) < 1e-6

assert abs(f_torch(torch.zeros(4)) + 4) < 1e-6
assert abs(f_torch(torch.ones(2)) - 1) < 1e-6

x0 = (1, -2, 3, 4, 5)
assert abs(f_numpy(np.array(x0)) - f_torch(torch.Tensor(x0))) < 1e-6

### Problem 2: Computing gradients manually and automatically
Implement functions to compute the gradient of $f$ at $x$ using numpy and PyTorch. Use autodiff for the latter.

In [25]:
def f_grad_numpy(x, h: float = 1e-6):
    """
    Numerical gradient of f using centered finite differences.
    x: numpy array
    h: step size
    """
    x = np.asarray(x, dtype=float)
    g = np.empty_like(x)
    for i in range(x.size):
        ei = np.zeros_like(x); ei[i] = 1.0
        g[i] = (f_numpy(x + h*ei) - f_numpy(x - h*ei)) / (2*h)
    return g

def f_grad_torch(x):
    # ∇f(x) via autograd
    f = 3 * torch.dot(x, x) - x[0] - 4
    return torch.autograd.grad(f, x)[0]

In [26]:
def finite_diff(x, v):
    return (f_numpy(x + v) - f_numpy(x - v)) / (2 * np.linalg.norm(v))


x = np.ones(2)
v0 = 1e-4 * np.array([1, 0])
assert abs(finite_diff(x, v0) - f_grad_numpy(x)[0]) < 1e-2
v1 = 1e-4 * np.array([0, 1])
assert abs(finite_diff(x, v1) - f_grad_numpy(x)[1]) < 1e-2

np.random.seed(42)
for i in range(10):
    x2 = np.random.randn(5)
    v2 = np.random.randn(5)
    v2 = v2 / np.linalg.norm(v2)
    observed = finite_diff(x2, 1e-4 * v2)
    derived_vec = f_grad_numpy(x2)
    derived = derived_vec.dot(v2)
    assert abs(observed - derived) < 1e-2

xt = torch.tensor(x, requires_grad=True)
diff = f_grad_torch(xt).numpy() - f_grad_numpy(x)
assert np.linalg.norm(diff) < 1e-6

In [ ]:
# Single layer from scratch 
import torch 
import torch.nn as nn 
import torch.nn.functional as F 

class MyDenseLayer(nn.Module): 
    def __init__(self, input_dim, output_dim): 
        super(MyDenseLayer, self).__init__()

        # Initialize weights and biases 
        self.weights = nn.Parameter(torch.randn(output_dim, input_dim))
        self.bias = nn.Parameter(torch.randn(output_dim))
        # self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x): 
        # perform matrix multiplication and add bias
        z = torch.matmul(x, self.weights.t()) + self.bias

        # apply ReLU activation function 
        y = F.relu(z) 
        return y 

In [2]:
# Multi-layer perceptron (two layer DNN)

class MultilayerPerceptron(nn.Module): 
    def __init__(self, num_features, hidden_size1, hidden_size2, num_classes): 
        super(MultilayerPerceptron, self).__init__()

        # hidden layers 
        self.linear_1 = nn.Linear(num_features, hidden_size1)
        self.linear_2 = nn.Linear(hidden_size1, hidden_size2)

        # output layer 
        self.linear_out = nn.Linear(hidden_size2, num_classes)
    
    def forward(self, x):
        x = F.relu(self.linear_1(x))
        x = F.relu(self.linear_2(x))
        logits = self.linear_out(x)
        probas = F.softmax(logits, dim=1)
        return logits, probas

## Additional Practice

In [1]:
import numpy as np

# N is batch size: D_in is input dimension 
# H is hidden dimension: D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# Create random input and output data 
x = np.random.randn(N, D_in)
y = np.random.randn(N, D_out)

# randomly initialize weights 
w1 = np.random.randn(D_in, H)
w2 = np.random.randn(H, D_out)

learning_rate = 1e-6
for t in range(500): 
    # forward pass: compute predicted y 
    h = x.dot(w1)
    h_relu = np.maximum(h, 0)
    y_pred = h_relu.dot(w2)

    # compute and print loss 
    loss = np.square(y_pred - y).sum()
    print(t, loss)

    # Backprop to compute gradients of w1 and w2 with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_w2 = h_relu.T.dot(grad_y_pred)
    grad_h_relu = grad_y_pred.dot(w2.T)
    grad_h = grad_h_relu.copy()
    grad_h[h < 0] = 0
    grad_w1 = x.T.dot(grad_h)

    # Update weights 
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2


0 29897296.430381544
1 26962584.47045411
2 27468628.149254657
3 27390384.179953452
4 24274210.8716707
5 18236242.52621002
6 11677670.621715702
7 6741716.3456271775
8 3804810.674696991
9 2254119.401529695
10 1458925.8130345303
11 1035952.2234913146
12 792379.6196006655
13 637789.9398676101
14 530269.4166423958
15 450177.46631961915
16 387324.08938689163
17 336332.46184837294
18 294102.41759266367
19 258666.54355781054
20 228588.88808150546
21 202870.6906164778
22 180736.62134365013
23 161566.68545125017
24 144883.2439612173
25 130286.56434169508
26 117455.20340207385
27 106159.17156372005
28 96168.2305942708
29 87302.78499309298
30 79405.5756901405
31 72353.3355266196
32 66039.47018651046
33 60373.63238218565
34 55278.46313021333
35 50683.67453970069
36 46532.35632960438
37 42775.68527884986
38 39370.416172968726
39 36276.99993906291
40 33461.59183106846
41 30896.742899505003
42 28557.09752515346
43 26416.612414558993
44 24458.965081343315
45 22665.248633379415
46 21019.857336466546
47 

In [3]:
import torch 

dtype = torch.FloatTensor 
# dtype = torch.cuda.FloatTensor # Uncomment this to run on GPU

# N is batch size: D_in is input dimension 
# H is hidden dimension: D_out is output dimension 
N, D_in, H, D_out = 64, 1000, 100, 10

# Create random input and output data
x = torch.randn(N, D_in).type(dtype)
y = torch.randn(N, D_out).type(dtype)

# Randomly initialize weights 
w1 = torch.randn(D_in, H).type(dtype)
w2 = torch.randn(H, D_out).type(dtype)

learning_rate = 1e-6
for t in range(500): 
    # forward pass: compute predicted y 
    h = x.mm(w1)
    h_relu = h.clamp(min=0)
    y_pred = h_relu.mm(w2)

    # compute and print loss 
    loss = (y_pred - y).pow(2).sum()
    print(t, loss.item())

    # Backprop to compute gradients of w1 and w2 with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_w2 = h_relu.t().mm(grad_y_pred)
    grad_h_relu = grad_y_pred.mm(w2.t())
    grad_h = grad_h_relu.clone()
    grad_h[h < 0] = 0
    grad_w1 = x.t().mm(grad_h)

    # Update weights 
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2


0 25489752.0
1 20361008.0
2 20227564.0
3 22472354.0
4 24800526.0
5 24989304.0
6 21535646.0
7 15651664.0
8 9725423.0
9 5522133.0
10 3070101.0
11 1793575.5
12 1142452.75
13 801457.875
14 609246.625
15 490018.40625
16 408428.4375
17 347898.9375
18 300358.40625
19 261568.390625
20 229146.5625
21 201656.03125
22 178140.75
23 157869.5
24 140294.84375
25 124995.5625
26 111629.390625
27 99926.6640625
28 89647.0078125
29 80593.7109375
30 72600.2890625
31 65517.46484375
32 59229.0859375
33 53635.71875
34 48649.5546875
35 44196.12109375
36 40215.546875
37 36653.17578125
38 33453.81640625
39 30574.296875
40 27978.76953125
41 25635.892578125
42 23516.9140625
43 21597.71875
44 19859.8828125
45 18282.1640625
46 16847.654296875
47 15541.7294921875
48 14351.8525390625
49 13265.9912109375
50 12273.9404296875
51 11366.2109375
52 10535.1865234375
53 9773.5859375
54 9075.07421875
55 8434.7646484375
56 7846.2646484375
57 7304.599609375
58 6805.85400390625
59 6345.8544921875
60 5921.43212890625
61 5529.39501

In [4]:
# Properties of Tensors 
new = torch.tensor([[1,2], [3,4]])
print(new)
print(new.dtype)
print(new.shape)
print(new.device) # can also be supported on GPU if available

tensor([[1, 2],
        [3, 4]])
torch.int64
torch.Size([2, 2])
cpu


In [5]:
# Creating Tensors 

# with built-in functions
x = torch.empty(5, 3) # uninitialized values
print(x)
x1 = torch.rand(5, 3) # random values between 0 and 1
print(x1)
x2 = torch.zeros(5, 3, dtype=torch.long) # zeros with long integer type
print(x2)
x3 = torch.tensor([5.0, 3]) # from data
print(x3); print(x3.shape); print(x3.dtype); print(x3.device)


tensor([[-6.4554e+06,  2.1286e-42,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00]])
tensor([[0.5841, 0.7554, 0.0222],
        [0.1823, 0.4868, 0.1892],
        [0.8437, 0.7624, 0.7432],
        [0.0657, 0.2834, 0.0104],
        [0.2238, 0.7914, 0.1545]])
tensor([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]])
tensor([5., 3.])
torch.Size([2])
torch.float32
cpu


In [6]:
# Creating tensors 

# based on existing tensor 
x = x.new_ones(5, 3, dtype = torch.double)
print(x)

# same shape; override value and data type 
x = torch.rand_like(x, dtype=torch.float)
print(x); print(x.dtype)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]], dtype=torch.float64)
tensor([[0.6576, 0.4274, 0.5076],
        [0.3401, 0.4681, 0.5661],
        [0.8342, 0.9246, 0.3156],
        [0.2319, 0.7571, 0.4399],
        [0.0666, 0.4779, 0.6242]])
torch.float32


In [7]:
# Operation examples: add 
x = torch.empty(5, 3)
y = torch.rand(5, 3)
z = x + y 
z1 = torch.add(x, y)
z2 = torch.empty(5, 3)
torch.add(x, y, out=z2)
y.add(x) ; # add in-place; modify operated variable
z3 = y

print(z); print(z1); print(z2); print(z3)


tensor([[-6.4567e+06,  6.8228e-01,  2.2409e-01],
        [ 9.9020e-01,  4.2115e-01,  6.3221e-01],
        [ 8.7341e-01,  4.1585e-01,  4.4762e-01],
        [ 8.4396e-01,  8.7262e-01,  6.2174e-01],
        [ 5.5792e-01,  8.7335e-01,  4.6287e-01]])
tensor([[-6.4567e+06,  6.8228e-01,  2.2409e-01],
        [ 9.9020e-01,  4.2115e-01,  6.3221e-01],
        [ 8.7341e-01,  4.1585e-01,  4.4762e-01],
        [ 8.4396e-01,  8.7262e-01,  6.2174e-01],
        [ 5.5792e-01,  8.7335e-01,  4.6287e-01]])
tensor([[-6.4567e+06,  6.8228e-01,  2.2409e-01],
        [ 9.9020e-01,  4.2115e-01,  6.3221e-01],
        [ 8.7341e-01,  4.1585e-01,  4.4762e-01],
        [ 8.4396e-01,  8.7262e-01,  6.2174e-01],
        [ 5.5792e-01,  8.7335e-01,  4.6287e-01]])
tensor([[0.8181, 0.6823, 0.2241],
        [0.9902, 0.4212, 0.6322],
        [0.8734, 0.4158, 0.4476],
        [0.8440, 0.8726, 0.6217],
        [0.5579, 0.8733, 0.4629]])


In [8]:
# Operation examples: reshape 
x = torch.rand(4, 4)
y = x.view(16)
z = x.reshape(-1, 8); # -1 infers dimension size
print(x.shape); print(y.shape); print(z.shape)



torch.Size([4, 4])
torch.Size([16])
torch.Size([2, 8])


In [9]:
# Tensor to and from a NumPy Array 

a = torch.ones(5, dtype = torch.float64)
b = a.numpy()
print(a); print(type(a)); print(b); print(type(b))

tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
<class 'torch.Tensor'>
[1. 1. 1. 1. 1.]
<class 'numpy.ndarray'>


In [10]:
# CUDA tensors 
    # Compute Unified Device Architecture 
    # NVIDIA's parallel computing platform and programming model that allows developers to use NVIDIA GPUs for general purpose computing 

# let us run this cell only if CUDA is available 
# use `torch.device` objects to move tensors in and out of GPU 
if torch.cuda.is_available(): 
    device = torch.device("cuda") # CUDA device object
    y = torch.ones_like(x, device=device) # directly create a tensor on GPU
    x = x.to(device) # or use `.to("cuda")`
    z = x + y
    print(z)
    print(z.to("cpu", torch.double)) # `.to` can also change dtype together


In [ ]:
# # GPU speedup on Google colab 
# import torch 
# import time 
# x_cpu = torch.randn(120000, 10000) 
# y_cpu = torch.randn(10000, 1)

# x_gpu = x_cpu.cuda() 
# y_gpu = y_cpu.cuda()

# start = time.time() 
# x_cpu.mm(y_cpu)
# print(time.time() - start)

# start = time.time()
# x_gpu.mm(y_gpu)
# print(time.time() - start)

In [11]:
# Automatic Differentiation 

# requires_grad = True to track all operations on this tensor; to compute gradients w.r.t it later 
x = torch.ones(2, 2, requires_grad = True); print(x)

y = x + 2; print(y)
# reference to the function that generated y; how to compute derivatives for addition during backpropagation
print(y.grad_fn) # gradient function that created y

z = y * y + 2; print(z)
out = z.mean()
print(z, out)

tensor([[1., 1.],
        [1., 1.]], requires_grad=True)
tensor([[3., 3.],
        [3., 3.]], grad_fn=<AddBackward0>)
tensor([[11., 11.],
        [11., 11.]], grad_fn=<AddBackward0>)
tensor([[11., 11.],
        [11., 11.]], grad_fn=<AddBackward0>) tensor(11., grad_fn=<MeanBackward0>)


In [12]:
# Gradients 
x = torch.tensor([1.0], requires_grad=True)
y = x * 2; print(y) # grad_fn = <MulBackward0>
z = y + 3; print(z) # grad_fn = <AddBackward0>

# when we call backward(), PyTorch uses these grad_fns to compute: 
z.backward()
print(x.grad) # dz/dx = (dz/dy) * (dy/dx) = 1 * 2 = 2


tensor([2.], grad_fn=<MulBackward0>)
tensor([5.], grad_fn=<AddBackward0>)
tensor([2.])


In [ ]:
# Autograd 

# simple autograd 
x = torch.tensor([2.0], requires_grad=True)
y = x * 2
y.backward()
print(x.grad) # 2.0

# multiple operations 
x = torch.tensor([2.0], requires_grad=True)
y = x * 2
z = 2 * y + 1 
z.backward()
print(x.grad) # 4.0; dz/dx = dz/dy * dy/dx = 2 * 2 = 4

# using torch.autograd.grad() directly 
    # alternative to backward() 
x = torch.tensor([2.0], requires_grad=True)
y = x * 2
gradient = torch.autograd.grad(y, x)[0]
print(gradient) # 2.0


tensor([2.])
tensor([4.])
tensor([2.])


In [ ]:
# Buld a single layer from scratch 
import torch 
import torch.nn as nn 
import torch.nn.functional as F 

class MyDenseLayer(nn.Module): 
    def __init__(self, input_dim, output_dim): 
        super(MyDenseLayer, self).__init__()

        # Initialize weights and biases
        self.weights = nn.Parameter(torch.randn(output_dim, input_dim))
        self.bias = nn.Parameter(torch.randn(output_dim))

    def forward(self, x): 
        # perform matrix multiplication and add bias
        z = torch.matmul(x, self.weights.t()) + self.bias

        # apply ReLU activation function 
        y = F.relu(z) 
        return y


In [ ]:
# Multi-layer perceptron (two layer DNN)
    # classification task
class MultilayerPerceptron(nn.Module): 
    def __init__(self, num_features, hidden_size1, hidden_size2, num_classes): 
        super(MultilayerPerceptron, self).__init__()

        # hidden layers 
        self.linear_1 = nn.Linear(num_features, hidden_size1)
        self.linear_2 = nn.Linear(hidden_size1, hidden_size2)

        # output layer 
        self.linear_out = nn.Linear(hidden_size2, num_classes)
    
    def forward(self, x): 
        x = F.relu(self.linear_1(x))
        x = F.relu(self.linear_2(x))
        logits = self.linear_out(x)
        probas = F.softmax(logits, dim=1)
        return logits, probas